# Haiku Linear Probing Accuracy

**Project Name:** Haiku (renamed from Haiku)

## Purpose
- Publication-ready notebook for reproducible training/evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

HAIKU_ROOT = Path('/home/yancui/Haiku')
if str(HAIKU_ROOT / 'src') not in sys.path:

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root='/home/yancui/Haiku')
seed_everything(42)


In [ ]:
%load_ext cuml.accel

In [ ]:
import json
import pandas as pd



#sample_dict = json.load(open('/home/yancui/Haiku/overlap_samples.json'))
sample_dict = json.load(open('/home/yancui/Haiku/src/training/overlap_samples_final.json'))
# sample_ids = sample_ids = pd.read_csv('/project/zhihuanglab/jleiby/codex_clip/sample_ids_with_text.csv', header=None)[0].tolist()
sample_ids = list(sample_dict.keys())


with open('/home/yancui/OmicsAnnotator/test_regions.txt', 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

#sample_ids = test_ids

holdout_list = pd.read_csv('/home/yancui/tier2_acquisition_ids_huanglab (3).csv')['ACQUISITION_ID'].tolist()

overlap_holdout_list = list(set(holdout_list) & set(sample_ids))

new_holdout_list = list(json.load(open('/home/yancui/Haiku/overlap_samples_new.json')).keys())

sample_ids = list(set(test_ids + list(set(overlap_holdout_list) - set(new_holdout_list))))

ref_ids = sorted(sample_ids)

In [ ]:

import os
from tqdm import tqdm

region_metadata_dir = "/data/enable_data/region_metadata"

# First, read all region_metadata CSVs into a dict: {region_id: df}
region_metadata = {}
metadata_files = [fname for fname in os.listdir(region_metadata_dir) if fname.endswith('.metadata.csv')]
print(f"Reading {len(metadata_files)} region metadata CSVs...")
for fname in tqdm(metadata_files, desc="Reading region metadata", total=len(metadata_files)):
    region_id = fname.split('.')[0]
    try:
        df = pd.read_csv(os.path.join(region_metadata_dir, fname))
        region_metadata[region_id] = df
    except Exception as e:
        print(f"Error reading {fname}: {e}")


In [ ]:
import torch

he_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/he_embedding.pt')
codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/codex_embedding.pt')
region_label = torch.load('/home/yancui/Haiku/res_embdding_126/region_label.pt')
virtual_codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/virtual_codex_embedding.pt')
text_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/text_embedding.pt')
musk_he_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/musk_he_embedding.pt')

In [ ]:
from tqdm import tqdm
import numpy as np

ref_ids = sorted(sample_ids)

metadata_dict_values = {}
metadata_dict_ids = {}

keys = ['tissue_type', 'grade', 'tnm', 'type']

for key in keys:
    metadata_dict_values[key] = []
    metadata_dict_ids[key] = []

print(f"Processing {len(region_label)} samples for metadata lookup...")
for i, sample in tqdm(enumerate(region_label), total=len(region_label), desc="Processing metadata"):
    #patch_id = sample['patch_id']
    #print(sample)
    region_id = ref_ids[sample].split('_')[0]
    df = region_metadata.get(region_id, None)
    if df is not None:
        for key in keys:
            if key in df['FEATURE_NAME'].values:
                if (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'nan') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'Unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == np.nan):
                    continue
                else:
                    metadata_dict_values[key].append(df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0])
                    metadata_dict_ids[key].append(i)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

def linear_probe_kfold(
    embeddings,
    labels,
    n_splits=5,
    s=5,
    random_state=42,
    Cs=None,
    select_by="f1_macro"  # one of {"f1_macro", "f1_micro", "accuracy"}
):
    """
    Perform linear probing with logistic regression using K-fold cross-validation,
    grid-searching over C (default 5 values) and picking the best C by mean metric.

    Returns the SAME flat dict as the previous version:
        {
            "f1_macro_mean", "f1_macro_std",
            "f1_micro_mean", "f1_micro_std",
            "accuracy_mean", "accuracy_std",
            "f1_macros", "f1_micros", "accuracies"
        }
    """
    # Convert tensors → numpy
    if hasattr(embeddings, "cpu"):
        embeddings = embeddings.cpu().numpy()
    if hasattr(labels, "cpu"):
        labels = labels.cpu().numpy()
    embeddings = np.asarray(embeddings)
    labels = np.asarray(labels)

    # Default 5-point C grid if not given
    if Cs is None:
        Cs = np.logspace(-2, 2, 5)  # [0.01, 0.1, 1, 10, 100]
    Cs = np.asarray(Cs, dtype=float)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # Evaluate each C with the same outer CV
    summaries = {}
    for C in Cs:
        f1_macros, f1_micros, accuracies = [], [], []
        for train_idx, test_idx in skf.split(embeddings, labels):
            X_train, X_test = embeddings[train_idx], embeddings[test_idx]
            y_train, y_test = labels[train_idx], labels[test_idx]

            clf = LogisticRegression(
                C=C,
                random_state=42,
                max_iter=50000,
                solver="lbfgs",
                multi_class="auto",
            )
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)

            f1_macros.append(f1_score(y_test, y_pred, average='macro', zero_division=0))
            f1_micros.append(f1_score(y_test, y_pred, average='micro', zero_division=0))
            accuracies.append(accuracy_score(y_test, y_pred))

        summaries[float(C)] = {
            "f1_macro_mean": np.mean(f1_macros),
            "f1_macro_std":  np.std(f1_macros),
            "f1_micro_mean": np.mean(f1_micros),
            "f1_micro_std":  np.std(f1_micros),
            "accuracy_mean": np.mean(accuracies),
            "accuracy_std":  np.std(accuracies),
            "f1_macros": f1_macros,
            "f1_micros": f1_micros,
            "accuracies": accuracies,
        }

    # Pick best C by requested metric
    key_mean = f"{select_by}_mean"
    best_C = max(summaries.keys(), key=lambda c: summaries[c][key_mean])
    best_res = summaries[best_C]

    # Pretty print
    print("=== Linear Probe K-Fold with C Grid Search ===")
    print(f"C candidates: {list(map(float, Cs))}")
    print(f"Selection metric: {select_by} (mean across folds)\n")
    for C in Cs:
        r = summaries[float(C)]
        print(f"C={float(C):g}  |  "
              f"F1 Macro: {r['f1_macro_mean']:.4f} ± {r['f1_macro_std']:.4f}  |  "
              f"F1 Micro: {r['f1_micro_mean']:.4f} ± {r['f1_micro_std']:.4f}  |  "
              f"Acc: {r['accuracy_mean']:.4f} ± {r['accuracy_std']:.4f}")
    print("\n--- Best ---")
    print(f"Best C: {best_C:g}")
    print(f"F1 Macro: {best_res['f1_macro_mean']:.4f} ± {best_res['f1_macro_std']:.4f}")
    print(f"F1 Micro: {best_res['f1_micro_mean']:.4f} ± {best_res['f1_micro_std']:.4f}")
    print(f"Accuracy: {best_res['accuracy_mean']:.4f} ± {best_res['accuracy_std']:.4f}")

    # Return EXACTLY like previous
    results = {
        "f1_macro_mean": best_res["f1_macro_mean"],
        "f1_macro_std":  best_res["f1_macro_std"],
        "f1_micro_mean": best_res["f1_micro_mean"],
        "f1_micro_std":  best_res["f1_micro_std"],
        "accuracy_mean": best_res["accuracy_mean"],
        "accuracy_std":  best_res["accuracy_std"],
        "f1_macros":     best_res["f1_macros"],
        "f1_micros":     best_res["f1_micros"],
        "accuracies":    best_res["accuracies"],
    }

    print("=== Linear Probe Results (selected C) ===")
    print(f"F1 Macro: {results['f1_macro_mean']:.4f} ± {results['f1_macro_std']:.4f}")
    print(f"F1 Micro: {results['f1_micro_mean']:.4f} ± {results['f1_micro_std']:.4f}")
    print(f"Accuracy: {results['accuracy_mean']:.4f} ± {results['accuracy_std']:.4f}")
    return results

def majority_vote_baseline(labels, n_splits=5, random_state=42):
    """
    Majority Voting Baseline: for each fold, predict the most frequent class
    in the training split for all test examples.

    Returns:
        {
            "f1_macro_mean", "f1_macro_std",
            "f1_micro_mean", "f1_micro_std",
            "accuracy_mean", "accuracy_std",
            "f1_macros", "f1_micros", "accuracies"
        }
    """
    if hasattr(labels, "cpu"):
        labels = labels.cpu().numpy()
    labels = np.asarray(labels)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    f1_macros, f1_micros, accuracies = [], [], []
    for train_idx, test_idx in skf.split(np.zeros_like(labels), labels):
        y_train, y_test = labels[train_idx], labels[test_idx]
        vals, counts = np.unique(y_train, return_counts=True)
        majority_class = vals[np.argmax(counts)]
        majority_pred = np.full_like(y_test, majority_class)
        f1_macros.append(f1_score(y_test, majority_pred, average='macro', zero_division=0))
        f1_micros.append(f1_score(y_test, majority_pred, average='micro', zero_division=0))
        accuracies.append(accuracy_score(y_test, majority_pred))
    results = {
        "f1_macro_mean": np.mean(f1_macros),
        "f1_macro_std":  np.std(f1_macros),
        "f1_micro_mean": np.mean(f1_micros),
        "f1_micro_std":  np.std(f1_micros),
        "accuracy_mean": np.mean(accuracies),
        "accuracy_std":  np.std(accuracies),
        "f1_macros": f1_macros,
        "f1_micros": f1_micros,
        "accuracies": accuracies,
    }
    print("=== Majority Voting Baseline ===")
    print(f"F1 Macro: {results['f1_macro_mean']:.4f} ± {results['f1_macro_std']:.4f}")
    print(f"F1 Micro: {results['f1_micro_mean']:.4f} ± {results['f1_micro_std']:.4f}")
    print(f"Accuracy: {results['accuracy_mean']:.4f} ± {results['accuracy_std']:.4f}")
    return results


In [ ]:
np.unique(metadata_dict_values['grade'])

In [ ]:
np.unique(metadata_dict_values['type'])

In [ ]:
np.unique(metadata_dict_values['tnm'])

In [ ]:
category_map = {}

category_map['type'] = {
    'normal': 'Normal',
    'nat': 'Normal',
    'at': 'Normal',
    'AT': 'Normal',
    "NAT": 'Normal',
    'hyperplasia': 'Benign/Precancerous',
    'malignant': 'Primary Tumor',
    'tumor primary': 'Primary Tumor',
    'Tumor Primary': 'Primary Tumor',
    'Maglignant': 'Primary Tumor',
    'metastasis': 'Metastatic Tumor',
    'nan': None,
    '-': None,
    '*': None
}

category_map['grade'] = {
    '1': 'G1',
    '1--2': 'G1',
    'g1': 'G1',
    '2': 'G2',
    '2--3': 'G2',
    'g2': 'G2',
    '3': 'G3',
    'g3': 'G3',
    'nan': None,
    '-': None,
    '*': None
}


In [ ]:
keys = ['tissue_type', 'grade', 'tnm', 'type']


def clean_labels(labels, map):
    """
    Map various survival status labels to 'alive' or 'death'.
    """
    mapped = []
    for x in labels:
        if x in map:
            mapped.append(map[x])
        else:
            mapped.append(x)

    return np.array(mapped)


for key in metadata_dict_values:
    filtered_values = []
    filtered_ids = []
    for val, idx in zip(metadata_dict_values[key], metadata_dict_ids[key]):
        if (
            val is not None
            and str(val).lower() != 'nan'
            and str(val).lower() != 'unknown'
            and str(val).lower() != '-'
            and not (isinstance(val, float) and np.isnan(val))
        ):
            filtered_values.append(val)
            filtered_ids.append(idx)
    metadata_dict_values[key] = filtered_values
    metadata_dict_ids[key] = filtered_ids


for key in metadata_dict_values:
    if key in category_map:
        filtered_values = clean_labels(metadata_dict_values[key], category_map[key])
        metadata_dict_ids[key] = np.array(metadata_dict_ids[key])[filtered_values != None]
        metadata_dict_values[key] = np.array(filtered_values)[filtered_values != None]


In [ ]:
import re

def split_tnm(values):
    """
    Split TNM strings like 'T3N1bM0' into separate labels.
    Filters out '-', 'nan', 'unknown', empty, None.

    Parameters
    ----------
    values : Sequence[str]
        Array/list of TNM strings.

    Returns
    -------
    result : dict
        {
          'indices': [idx_of_each_valid_entry],
          'T': ['T1', 'T1a', 'T4B', ...],
          'N': ['N0', 'N1b', ...],
          'M': ['M0', 'M1', ...]
        }
    """
    clean_exclude = {'-', '', 'nan', 'none', 'unknown'}
    # Capture contiguous T..., N..., M... chunks (letters+digits), in order.
    pat = re.compile(r'^(T[0-9A-Za-z]+)(N[0-9A-Za-z]+)(M[0-9A-Za-z]+)$')

    idxs, Ts, Ns, Ms = [], [], [], []
    for i, v in enumerate(values):
        if v is None:
            continue
        s = str(v).strip()
        if s.lower() in clean_exclude:
            continue
        s = s.replace(' ', '')  # ensure no spaces like 'T3 N1 M0'
        m = pat.match(s)
        if not m:
            # Skip anything that doesn't look like proper TNM
            continue
        t, n, mstage = m.group(1), m.group(2), m.group(3)
        idxs.append(i)
        Ts.append(t)
        Ns.append(n)
        Ms.append(mstage)

    return {'indices': idxs, 'T': Ts, 'N': Ns, 'M': Ms}


res = split_tnm(metadata_dict_values['tnm'])

# assuming you want new keys 'T', 'N', 'M' parallel to 'tnm'
metadata_dict_values['T'] = np.array(res['T'])
metadata_dict_values['N'] = np.array(res['N'])
metadata_dict_values['M'] = np.array(res['M'])

# Map split_tnm positions back to original embedding indices via metadata_dict_ids['tnm']
tnm_ids = np.array(metadata_dict_ids['tnm'])
metadata_dict_ids['T'] = tnm_ids[res['indices']]
metadata_dict_ids['N'] = tnm_ids[res['indices']]
metadata_dict_ids['M'] = tnm_ids[res['indices']]

In [ ]:
cancer_indices = metadata_dict_ids['type'][(metadata_dict_values['type'] != 'Normal') & (metadata_dict_values['type'] != 'Benign/Precancerous')]

In [ ]:
np.unique(metadata_dict_values['type'][np.isin(metadata_dict_ids['type'], cancer_indices)])

In [ ]:
np.unique(metadata_dict_values['T'][np.isin(metadata_dict_ids['T'], cancer_indices)])

In [ ]:
print(np.unique(metadata_dict_values['grade'][np.isin(metadata_dict_ids['grade'], cancer_indices)]))


In [ ]:
majory_vote_results = {'Majority_Vote': {}}

key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

key = 'grade'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

key = 'T'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3
key = 'N'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_3 = majority_vote_baseline(label_1)

majory_vote_results['Majority_Vote'][key] = res_3

with open('linear_probing_results_majority_vote_126.json', 'w') as f:


In [ ]:
linear_probing_results = {'Our_CODEX': {}, 'Virtues': {}, 'Our_HE': {}, 'Musk': {}, 'Our_concat': {}}

In [ ]:
linear_probing_results

In [ ]:

key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2


key = 'grade'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2


key = 'T'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2

key = 'N'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2

key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(virtual_codex_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Virtues"))

linear_probing_results['Our_CODEX'][key] = res_1
linear_probing_results['Virtues'][key] = res_2


In [ ]:

key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2


key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1)
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1)

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

key = 'T'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

key = 'N'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

key = 'grade'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

res_1 = linear_probe_kfold(he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])
res_2 = linear_probe_kfold(musk_he_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

#plot_linear_probe_comparison(res_1, res_2, method_names=("Ours", "Musk"))

linear_probing_results['Our_HE'][key] = res_1
linear_probing_results['Musk'][key] = res_2

In [ ]:

key = 'tissue_type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

import numpy as np

# Concatenate he and codex embeddings along last axis
concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


linear_probing_results['Our_concat'][key] = res_1

#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))


key = 'grade'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


linear_probing_results['Our_concat'][key] = res_1
#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))

key = 'T'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

linear_probing_results['Our_concat'][key] = res_1

key = 'N'

id_1 = metadata_dict_ids[key][np.isin(metadata_dict_ids[key], cancer_indices)]
label_1 = metadata_dict_values[key][np.isin(metadata_dict_ids[key], cancer_indices)]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])


linear_probing_results['Our_concat'][key] = res_1

#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))


key = 'type'

id_1 = metadata_dict_ids[key]
label_1 = metadata_dict_values[key]

concat_embedding = np.concatenate([he_embedding, codex_embedding], axis=1)
res_1 = linear_probe_kfold(concat_embedding[id_1], label_1, Cs = [0.01, 0.1, 1.0, 10.0, 100.0])

linear_probing_results['Our_concat'][key] = res_1

#plot_linear_probe_comparison(res_1, res_2, method_names=("HE+Codex", "Virtues"))



In [ ]:
with open('linear_probing_results_new_126.json', 'w') as f:


In [ ]:
linear_probing_results

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import mannwhitneyu  # rank-sum test ✅

# =========================
# Style (Illustrator-friendly)
# =========================
cmap = ['#EAAA60', '#E68B81', '#90A4AE', '#7DA6C6', '#84C3B7', '#B7B2D0']

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['svg.fonttype'] = 'none'   # keep text editable
plt.rcParams['pdf.fonttype'] = 42       # embed TrueType
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 16

# =========================
# Load results
# =========================
with open('linear_probing_results_new_126.json', 'r') as f:
    linear_probing_results = json.load(f)

with open('linear_probing_results_majority_vote_126.json', 'r') as f:
    majory_vote_results = json.load(f)

# Merge in majority vote results
linear_probing_results.update(majory_vote_results)


# =========================
# Helpers
# =========================
def _extract_metric_samples(metric_dict, metric_key="f1_macro"):
    """
    Return replicate array for e.g. 'f1_macro' -> 'f1_macros'.
    """
    key = metric_key + "s"
    if key in metric_dict and isinstance(metric_dict[key], (list, tuple, np.ndarray)):
        xs = np.asarray(metric_dict[key], dtype=float)
        xs = xs[np.isfinite(xs)]
        return xs
    return None


def _extract_means_stds(metric_dict, metric_key="f1_macro"):
    """
    Prefer provided mean/std; else compute from replicates if present.
    """
    mk, sk = metric_key + "_mean", metric_key + "_std"
    if mk in metric_dict and sk in metric_dict:
        mean = float(metric_dict[mk])
        std = float(metric_dict[sk])
        return mean, std

    xs = _extract_metric_samples(metric_dict, metric_key)
    if xs is not None and xs.size > 0:
        mean = float(xs.mean())
        std = float(xs.std(ddof=1)) if xs.size >= 2 else 0.0
        return mean, std

    return np.nan, np.nan


def _compute_mwu_one_sided(results, metric, ours, other, dataset, alternative="greater"):
    """
    One-sided Mann–Whitney U: H1 = ours > other.
    Returns (U, p) or (None, None) if insufficient replicates.
    """
    our_x = _extract_metric_samples(results.get(ours, {}).get(dataset, {}), metric)
    oth_x = _extract_metric_samples(results.get(other, {}).get(dataset, {}), metric)

    if our_x is None or oth_x is None:
        return None, None
    if our_x.size < 2 or oth_x.size < 2:
        return None, None

    U, p = mannwhitneyu(our_x, oth_x, alternative=alternative)
    return float(U), float(p)


def _format_p_text(p, style="threshold"):
    """
    style='threshold': use 'P < 0.0001' style (like the example).
    style='exact': use 'P = 0.0032'.
    """
    if p is None or not np.isfinite(p):
        return None

    if style == "exact":
        # keep short, avoid scientific notation unless needed
        if p >= 0.001:
            return f"P = {p:.3f}".rstrip('0').rstrip('.')
        return f"P = {p:.2e}"

    # threshold style
    if p < 1e-4:
        return "P < 0.0001"
    if p < 1e-3:
        return "P < 0.001"
    if p < 1e-2:
        return "P < 0.01"
    if p < 5e-2:
        return "P < 0.05"
    return f"P = {p:.3f}".rstrip('0').rstrip('.')


def _add_pvalue_bracket(ax, x1, x2, y, text, h, lw=1.5, fs=14):
    """
    Draw a bracket from x1 to x2 at height y with bracket height h and centered label.
    """
    # bracket shape: └──┘
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='black', lw=lw, clip_on=False)
    ax.text((x1 + x2) / 2.0, y + h, text, ha='center', va='bottom',
            fontsize=fs, fontweight='bold', color='black', clip_on=False)


def _print_pairwise_pvals(pvals_dict):
    print("One-sided (greater) Mann–Whitney U p-values (ours > baseline):")
    for (ds, baseline), (U, p) in pvals_dict.items():
        ptxt = _format_p_text(p, style="exact")
        print(f"{ds:20s} vs {baseline:18s}  U={U:.2f}  {ptxt}")


# =========================
# Main plot
# =========================
def plot_grouped_rank_sum_with_second_best_brackets(
    results,
    metric="f1_macro",
    our_method="Our_concat",
    methods=None,
    save_path="figs/linear_probing_f1_macro_rank_sum.svg",
    bar_width=0.16,
    capsize=3,
    figsize=(14, 8),
    ylim=(0, 1),
    scale_to_percent=False,
    p_text_style="threshold",   # "threshold" -> "P < 0.0001" like example; "exact" -> "P = ..."
    bracket_pad_frac=0.02,      # vertical pad as fraction of y-range
    bracket_h_frac=0.01         # bracket height as fraction of y-range
):
    """
    Grouped bars across datasets with error bars.
    Adds ONE p-value bracket per dataset between our_method and the second-best method
    (among all non-our methods), matching the bracket+P-text style from the example figure.
    """
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)

    # Collect datasets
    datasets = sorted({d for v in results.values() for d in v.keys()})

    # Default methods order
    if methods is None:
        methods = ['Majority_Vote', 'Virtues', 'Musk', 'Our_HE', 'Our_CODEX', 'Our_concat']

    if our_method not in methods:
        raise ValueError(f"our_method '{our_method}' must be included in methods list.")

    # Colors
    colors = {m: cmap[i % len(cmap)] for i, m in enumerate(methods)}

    # Means/stds
    all_means, all_stds = {}, {}
    for m in methods:
        ms, ss = [], []
        for ds in datasets:
            mean, std = _extract_means_stds(results.get(m, {}).get(ds, {}), metric)

            # Optional scaling if your JSON stores [0,1] but you want %
            if scale_to_percent and np.isfinite(mean) and 0 <= mean <= 1:
                mean *= 100.0
                std *= 100.0

            ms.append(mean)
            ss.append(std)
        all_means[m], all_stds[m] = ms, ss

    # Print a compact mean table
    print(f"\nMean performance per method per task (metric: {metric}):")
    print("{:<18s}".format("Method/Task"), end="")
    for ds in datasets:
        print(" | {:<18s}".format(ds), end="")
    print()
    print("-"*18 + "+" + "+".join(["-"*20 for _ in datasets]))
    for m in methods:
        print("{:<18s}".format(m), end="")
        for j in range(len(datasets)):
            val = all_means[m][j]
            s = "   -   " if np.isnan(val) else (f"{val:7.4f}" if not scale_to_percent else f"{val:7.2f}%")
            print(" | {:<18s}".format(s), end="")
        print()
    print()

    # Draw bars
    fig, ax = plt.subplots(figsize=figsize)
    idx = np.arange(len(datasets))
    total_width = min(0.82, len(methods) * bar_width)
    start = -total_width / 2 + bar_width / 2

    bar_centers = {}  # bar_centers[(dataset_index, method)] = x
    for i, m in enumerate(methods):
        xs = idx + start + i * bar_width
        bar_centers.update({(j, m): xs[j] for j in range(len(datasets))})

        bars = ax.bar(
            xs, all_means[m], yerr=all_stds[m], width=bar_width,
            capsize=capsize, color=colors[m], label=m,
            edgecolor='black', linewidth=1
        )

        # Mean labels above bars
        yr = (ylim[1] - ylim[0])
        for j, rect in enumerate(bars):
            h = rect.get_height()
            if not np.isfinite(h):
                continue
            label_str = f"{h:.2f}%" if scale_to_percent else f"{h:.3f}"
            ax.text(
                rect.get_x() + rect.get_width() / 2.0,
                h + 0.01 * yr,
                label_str,
                ha='center', va='bottom',
                fontsize=12, fontweight='bold', color='black'
            )

    ax.set_xticks(idx)
    ax.set_xticklabels(datasets, rotation=35, ha="right")
    ax.set_ylabel(metric.replace("_", " ").title() + (" (%)" if scale_to_percent else ""))
    ax.set_ylim(*ylim)
    ax.legend()
    ax.set_title(f"{metric.replace('_',' ').title()} (MWU one-sided: ours > baseline)")

    # =========================
    # Add p-value bracket: Our vs second-best (per dataset)
    # =========================
    yr = (ylim[1] - ylim[0])
    pad = bracket_pad_frac * yr
    bh = bracket_h_frac * yr

    pvals_second_best = {}  # (dataset, second_best_method) -> (U, p)

    for j, ds in enumerate(datasets):
        # our mean/err
        our_mean = all_means[our_method][j]
        our_std = all_stds[our_method][j]

        if not np.isfinite(our_mean):
            continue

        # find second-best among non-our methods by mean (ties broken arbitrarily)
        candidates = [m for m in methods if m != our_method]
        cand_means = [(m, all_means[m][j]) for m in candidates if np.isfinite(all_means[m][j])]
        if len(cand_means) == 0:
            continue

        # second-best == best baseline (highest mean among baselines)
        second_best_method, second_best_mean = max(cand_means, key=lambda t: t[1])
        second_best_std = all_stds[second_best_method][j] if np.isfinite(all_stds[second_best_method][j]) else 0.0

        # compute MWU p-value (one-sided: ours > second_best)
        U, p = _compute_mwu_one_sided(results, metric, our_method, second_best_method, ds, alternative="greater")
        if p is None:
            continue

        pvals_second_best[(ds, second_best_method)] = (U, p)

        # bracket geometry
        x1 = bar_centers[(j, second_best_method)]
        x2 = bar_centers[(j, our_method)]
        if x1 > x2:
            x1, x2 = x2, x1

        top1 = our_mean + (our_std if np.isfinite(our_std) else 0.0)
        top2 = second_best_mean + (second_best_std if np.isfinite(second_best_std) else 0.0)
        y = max(top1, top2) + pad

        # p text like the example
        p_text = _format_p_text(p, style=p_text_style)
        if p_text is None:
            continue

        _add_pvalue_bracket(ax, x1, x2, y, p_text, h=bh, lw=1.5, fs=14)

    plt.tight_layout()

    # Save as SVG (dpi not needed for vector)
    plt.show()

    _print_pairwise_pvals(pvals_second_best)
    return pvals_second_best


# =========================
# Run
# =========================
methods_all = ['Majority_Vote', 'Virtues', 'Musk', 'Our_HE', 'Our_CODEX', 'Our_concat']

pvals = plot_grouped_rank_sum_with_second_best_brackets(
    linear_probing_results,
    metric="f1_macro",
    our_method="Our_concat",
    methods=methods_all,
    save_path="figs/linear_probing_f1_macro_rank_sum.svg",
    bar_width=0.16,
    capsize=3,
    figsize=(14, 8),
    ylim=(0, 1),
    scale_to_percent=False,
    p_text_style="threshold",   # <- produces "P < 0.0001" style like your example
    bracket_pad_frac=0.03,
    bracket_h_frac=0.012
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams['font.family'] = 'Arial'    # use Arial font
plt.rcParams['font.size'] = 12           # set large font size
plt.rcParams['font.weight'] = 'bold'     # make text bold
plt.rcParams['svg.fonttype'] = 'none'    # don't convert text to path


threshold = 0.05
outer_r   = 1.42
inner_r   = 0.68
fs_in     = 10
fs_out    = 8

for label, values in metadata_dict_values.items():
    if label == 'tnm':
        continue


    if (label != 'tissue_type') and (label != 'type'):
        values = values[np.isin(metadata_dict_ids[label], cancer_indices)]

    if isinstance(values, dict):
        total = sum(values.values())
        proportions = {k: (v / total if total > 0 else 0) for k, v in values.items()}
    else:
        counts = Counter(values)
        total = len(values)
        proportions = {k: (v / total if total > 0 else 0) for k, v in counts.items()}

    labels = np.array(list(proportions.keys()), dtype=object)
    sizes  = np.array(list(proportions.values()), dtype=float)

    big_idx   = np.where(sizes >= threshold)[0]
    small_idx = np.where(sizes <  threshold)[0]

    order = np.concatenate([big_idx, small_idx])

    labels = labels[order]
    sizes  = sizes[order]

    if label == 'type':
        labels[2], labels[4] = labels[4], labels[2]
        sizes[2], sizes[4] = sizes[4], sizes[2]

    if label == 'tissue_type':
        labels[-3], labels[-6] = labels[-6], labels[-3]
        sizes[-3], sizes[-6] = sizes[-6], sizes[-3]

    fig, ax = plt.subplots(figsize=(15, 10))

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=None,
        startangle=90,
        counterclock=False,
        colors=plt.get_cmap('tab20').colors,
        wedgeprops=dict(linewidth=0.8, edgecolor='white'),
        pctdistance=inner_r,
        autopct='%1.1f%%'
    )

    for i, w in enumerate(wedges):
        pct = sizes[i]
        ang = (w.theta2 + w.theta1) / 2.0
        rad = np.deg2rad(ang)
        x, y = np.cos(rad), np.sin(rad)
        text_full = f"{labels[i]} {pct*100:.1f}%"

        rot = ang
        if 90 < rot < 270:
            rot += 180

        if pct < threshold:
            if autotexts[i] is not None:
                autotexts[i].set_visible(False)

            tx, ty = x * outer_r, y * outer_r
            ha = 'left' if x >= 0 else 'right'

            ax.annotate(
                text_full,
                xy=(x*0.98, y*0.98),
                xytext=(tx, ty),
                ha=ha, va='center',
                fontsize=fs_out,
                rotation=rot,
                rotation_mode='anchor',
                bbox=None,
                arrowprops=dict(
                    arrowstyle='-',
                    lw=1.0,
                    shrinkA=0, shrinkB=0,
                    connectionstyle='arc3,rad=0'
                )
            )
        else:
            autotexts[i].set_text(text_full)
            autotexts[i].set_fontsize(fs_in)
            autotexts[i].set_rotation(rot)
            autotexts[i].set_rotation_mode('anchor')
            autotexts[i].set_ha('center')
            autotexts[i].set_va('center')

    ax.axis('equal')
    plt.tight_layout()
    plt.show()
